# CQLite Real-World Use Cases

This notebook demonstrates practical patterns for using CQLite Python bindings in real-world data workflows.

**Topics Covered:**
1. Data Export Patterns (CSV, JSON/JSONL, Parquet, streaming)
2. Data Analytics Integration (pandas, polars)
3. Batch Processing (large datasets, progress reporting)
4. Error Handling Patterns
5. Multi-table Operations

**Prerequisites:**
- CQLite bindings installed (`maturin develop` in bindings/python/)
- Test data available (`bash test-data/scripts/fetch-datasets.sh`)

**Optional Dependencies:**
- pandas: DataFrame operations
- polars: Fast analytics
- pyarrow: Parquet export

In [ ]:
# Environment Setup
import os
import sys
import csv
import json
import time
import tempfile
import tracemalloc
from pathlib import Path
from datetime import datetime, date, time as dt_time, timedelta
from decimal import Decimal
from uuid import UUID
from ipaddress import IPv4Address, IPv6Address
from typing import Any, Iterator, Callable, Optional

# Import cqlite
import cqlite

print(f"CQLite version: {cqlite.version()}")

# Setup paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent
DATASETS_ROOT = Path(os.environ.get('CQLITE_DATASETS_ROOT', PROJECT_ROOT / 'test-data' / 'datasets'))
DATA_DIR = DATASETS_ROOT / 'sstables'

# Schema files
SCHEMA_BASIC = PROJECT_ROOT / 'test-data' / 'schemas' / 'basic-types.cql'
SCHEMA_COLLECTIONS = PROJECT_ROOT / 'test-data' / 'schemas' / 'collections.cql'
SCHEMA_TIMESERIES = PROJECT_ROOT / 'test-data' / 'schemas' / 'time-series.cql'
SCHEMA_WIDE_ROWS = PROJECT_ROOT / 'test-data' / 'schemas' / 'wide-rows.cql'

# Cross-platform temp directory
TEMP_DIR = Path(tempfile.gettempdir())

# Verify setup
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"Schemas exist: basic={SCHEMA_BASIC.exists()}, collections={SCHEMA_COLLECTIONS.exists()}")
print(f"Temp directory: {TEMP_DIR}")

# Check optional dependencies
PANDAS_AVAILABLE = False
POLARS_AVAILABLE = False
PYARROW_AVAILABLE = False

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
    print(f"pandas: {pd.__version__} (available)")
except ImportError:
    print("pandas: not installed (optional - install with: pip install pandas)")

try:
    import polars as pl
    POLARS_AVAILABLE = True
    print(f"polars: {pl.__version__} (available)")
except ImportError:
    print("polars: not installed (optional - install with: pip install polars)")

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    PYARROW_AVAILABLE = True
    print(f"pyarrow: {pa.__version__} (available)")
except ImportError:
    print("pyarrow: not installed (optional - install with: pip install pyarrow)")

## 1. Data Export Patterns

Export CQLite query results to various formats for downstream processing.

In [ ]:
# 1.1 Export to CSV
# Use case: Share data with spreadsheet tools or data warehouses

def json_serializer(obj: Any) -> Any:
    """Custom JSON serializer for CQL types."""
    if isinstance(obj, (datetime, date, dt_time)):
        return obj.isoformat()
    if isinstance(obj, timedelta):
        return str(obj)
    if isinstance(obj, UUID):
        return str(obj)
    if isinstance(obj, Decimal):
        return str(obj)
    if isinstance(obj, bytes):
        return f"0x{obj.hex()}"
    if isinstance(obj, frozenset):
        return list(obj)
    if isinstance(obj, (IPv4Address, IPv6Address)):
        return str(obj)
    raise TypeError(f"Cannot serialize {type(obj)}")

def export_to_csv(db, query: str, output_path: Path, limit: Optional[int] = None) -> int:
    """Export query results to CSV file.
    
    Args:
        db: CQLite database connection
        query: CQL SELECT query
        output_path: Path to output CSV file
        limit: Optional row limit for testing
    
    Returns:
        Number of rows exported
    """
    try:
        full_query = f"{query} LIMIT {limit}" if limit else query
        result = db.execute(full_query)
    except cqlite.CqliteError as e:
        print(f"Query failed: {e}")
        return 0
    
    if not result.rows:
        print("No rows to export")
        return 0
    
    # Get column names from first row
    columns = result.rows[0].keys()
    
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        
        for row in result.rows:
            # Convert complex types to strings for CSV
            row_dict = {}
            for k, v in row.to_dict().items():
                if isinstance(v, (list, dict, frozenset, bytes)):
                    row_dict[k] = json.dumps(v, default=json_serializer)
                elif isinstance(v, (IPv4Address, IPv6Address)):
                    row_dict[k] = str(v)
                elif v is None:
                    row_dict[k] = ""
                else:
                    row_dict[k] = v
            writer.writerow(row_dict)
    
    return len(result.rows)

# Example: Export simple_table to CSV
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    output_file = TEMP_DIR / "simple_table_export.csv"
    count = export_to_csv(
        db, 
        "SELECT id, name, age, salary, created FROM test_basic.simple_table",
        output_file,
        limit=100
    )
    print(f"Exported {count} rows to {output_file}")
    
    # Show first few lines
    print("\nFirst 3 lines:")
    with open(output_file) as f:
        for i, line in enumerate(f):
            if i < 3:
                print(f"  {line.strip()[:80]}..." if len(line) > 80 else f"  {line.strip()}")

In [ ]:
# 1.2 Export to JSON/JSONL
# Use case: Data interchange, API integration, streaming pipelines

def export_to_jsonl(db, query: str, output_path: Path, limit: Optional[int] = None) -> int:
    """Export query results to JSONL (one JSON object per line).
    
    JSONL is ideal for:
    - Streaming processing
    - Appending to existing files
    - Processing large datasets line by line
    """
    try:
        full_query = f"{query} LIMIT {limit}" if limit else query
        result = db.execute(full_query)
    except cqlite.CqliteError as e:
        print(f"Query failed: {e}")
        return 0
    
    with open(output_path, 'w', encoding='utf-8') as f:
        for row in result.rows:
            json_line = json.dumps(row.to_dict(), default=json_serializer)
            f.write(json_line + '\n')
    
    return len(result.rows)

# Example: Export sensor data to JSONL
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_TIMESERIES)) as db:
    output_file = TEMP_DIR / "sensor_data.jsonl"
    count = export_to_jsonl(
        db,
        "SELECT * FROM test_timeseries.sensor_data",
        output_file,
        limit=50
    )
    print(f"Exported {count} rows to JSONL: {output_file}")
    
    # Show first 2 lines
    print("\nFirst 2 records:")
    with open(output_file) as f:
        for i, line in enumerate(f):
            if i < 2:
                data = json.loads(line)
                print(f"  Row {i}: {list(data.keys())[:4]}...")

In [ ]:
# 1.3 Export to Parquet (requires pyarrow)
# Use case: Data lakes, Spark/Presto/Athena queries, columnar analytics

if PYARROW_AVAILABLE and PANDAS_AVAILABLE:
    def export_to_parquet(db, query: str, output_path: Path, limit: Optional[int] = None) -> int:
        """Export query results to Parquet format.
        
        Parquet is ideal for:
        - Data lake storage (S3, GCS, HDFS)
        - Columnar analytics (Spark, Presto, Athena)
        - Efficient compression and encoding
        """
        try:
            full_query = f"{query} LIMIT {limit}" if limit else query
            result = db.execute(full_query)
        except cqlite.CqliteError as e:
            print(f"Query failed: {e}")
            return 0
        
        if not result.rows:
            print("No rows to export")
            return 0
        
        # Convert to list of dicts with serializable types
        rows = []
        for row in result.rows:
            row_dict = {}
            for k, v in row.to_dict().items():
                if isinstance(v, UUID):
                    row_dict[k] = str(v)
                elif isinstance(v, bytes):
                    row_dict[k] = f"0x{v.hex()}"
                elif isinstance(v, frozenset):
                    row_dict[k] = list(v)
                elif isinstance(v, Decimal):
                    row_dict[k] = float(v)
                elif isinstance(v, (IPv4Address, IPv6Address)):
                    row_dict[k] = str(v)
                else:
                    row_dict[k] = v
            rows.append(row_dict)
        
        # Convert to pandas then parquet
        df = pd.DataFrame(rows)
        table = pa.Table.from_pandas(df)
        pq.write_table(table, output_path)
        
        return len(rows)
    
    # Example: Export to Parquet
    with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
        output_file = TEMP_DIR / "simple_table.parquet"
        count = export_to_parquet(
            db,
            "SELECT id, name, age, salary, active FROM test_basic.simple_table",
            output_file,
            limit=100
        )
        print(f"Exported {count} rows to Parquet: {output_file}")
        print(f"File size: {output_file.stat().st_size:,} bytes")
        
        # Read back to verify
        table = pq.read_table(output_file)
        print(f"Parquet schema: {table.schema.names[:5]}...")
else:
    print("Skipping Parquet example - pyarrow and/or pandas not installed")
    print("Install with: pip install pyarrow pandas")

In [ ]:
# 1.4 Streaming Export for Large Datasets
# Use case: Export millions of rows without loading all into memory

def streaming_export_csv(db, query: str, output_path: Path, 
                         batch_report_interval: int = 1000) -> int:
    """Stream query results directly to CSV without loading all in memory.
    
    Key benefits:
    - Constant memory usage regardless of table size
    - Progress reporting during export
    - Safe to use with very large tables
    """
    config = cqlite.StreamingConfig(buffer_size=512, chunk_size=5000)
    
    rows_written = 0
    columns_written = False
    
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = None
        
        for row in db.execute_streaming(query, config=config):
            row_dict = row.to_dict()
            
            # Initialize CSV writer with column names from first row
            if not columns_written:
                writer = csv.DictWriter(f, fieldnames=row_dict.keys())
                writer.writeheader()
                columns_written = True
            
            # Serialize complex types
            serialized = {}
            for k, v in row_dict.items():
                if isinstance(v, (list, dict, frozenset, bytes)):
                    serialized[k] = json.dumps(v, default=json_serializer)
                elif isinstance(v, (IPv4Address, IPv6Address)):
                    serialized[k] = str(v)
                elif v is None:
                    serialized[k] = ""
                else:
                    serialized[k] = v
            
            writer.writerow(serialized)
            rows_written += 1
            
            # Progress reporting
            if rows_written % batch_report_interval == 0:
                print(f"  Exported {rows_written:,} rows...", end='\r')
    
    print(f"\nCompleted: {rows_written:,} rows exported")
    return rows_written

# Example: Stream export all rows
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    output_file = TEMP_DIR / "simple_table_streaming.csv"
    count = streaming_export_csv(
        db,
        "SELECT * FROM test_basic.simple_table",
        output_file,
        batch_report_interval=200
    )
    print(f"File size: {output_file.stat().st_size:,} bytes")

## 2. Data Analytics Integration

Convert CQLite results to popular analytics frameworks for exploration and analysis.

In [ ]:
# 2.1 Convert to pandas DataFrame
# Use case: Data exploration, statistical analysis, visualization

if PANDAS_AVAILABLE:
    def to_pandas(db, query: str, limit: Optional[int] = None) -> "pd.DataFrame":
        """Convert query results to pandas DataFrame.
        
        Handles CQL type conversions:
        - UUID -> string (for easier manipulation)
        - bytes -> hex string
        - frozenset -> list
        - IPv4/IPv6Address -> string
        - Decimal preserved as-is
        
        Note: Loads entire result set into memory. For large datasets,
        consider using execute_streaming() with chunked DataFrame creation.
        """
        full_query = f"{query} LIMIT {limit}" if limit else query
        result = db.execute(full_query)
        
        rows = []
        for row in result.rows:
            row_dict = {}
            for k, v in row.to_dict().items():
                # Convert types for better pandas compatibility
                if isinstance(v, UUID):
                    row_dict[k] = str(v)
                elif isinstance(v, bytes):
                    row_dict[k] = f"0x{v.hex()}"
                elif isinstance(v, frozenset):
                    row_dict[k] = list(v)
                elif isinstance(v, (IPv4Address, IPv6Address)):
                    row_dict[k] = str(v)
                else:
                    row_dict[k] = v
            rows.append(row_dict)
        
        return pd.DataFrame(rows)
    
    # Example: Analytics on simple_table
    with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
        df = to_pandas(db, "SELECT name, age, salary, active FROM test_basic.simple_table")
        
        print(f"DataFrame shape: {df.shape}")
        print(f"\nColumn types:\n{df.dtypes}")
        print(f"\nBasic statistics:")
        print(df.describe())
        
        # Example analysis
        if 'active' in df.columns and 'salary' in df.columns:
            print(f"\nActive users: {df['active'].sum()}")
            print(f"Average salary: ${df['salary'].mean():,.2f}")
            print(f"Age range: {df['age'].min()} - {df['age'].max()}")
else:
    print("Skipping pandas example - pandas not installed")
    print("Install with: pip install pandas")

In [ ]:
# 2.2 Integration with polars (fast DataFrame library)
# Use case: High-performance analytics, lazy evaluation

if POLARS_AVAILABLE:
    def to_polars(db, query: str, limit: Optional[int] = None) -> "pl.DataFrame":
        """Convert query results to polars DataFrame.
        
        polars benefits:
        - Faster than pandas for large datasets
        - Lazy evaluation support
        - Better memory efficiency
        
        Note: Loads entire result set into memory. For large datasets,
        consider using execute_streaming() with chunked processing.
        """
        full_query = f"{query} LIMIT {limit}" if limit else query
        result = db.execute(full_query)
        
        rows = []
        for row in result.rows:
            row_dict = {}
            for k, v in row.to_dict().items():
                # Convert types for polars compatibility
                if isinstance(v, UUID):
                    row_dict[k] = str(v)
                elif isinstance(v, bytes):
                    row_dict[k] = f"0x{v.hex()}"
                elif isinstance(v, frozenset):
                    row_dict[k] = list(v)
                elif isinstance(v, Decimal):
                    row_dict[k] = float(v)  # polars prefers float
                elif isinstance(v, (IPv4Address, IPv6Address)):
                    row_dict[k] = str(v)
                else:
                    row_dict[k] = v
            rows.append(row_dict)
        
        return pl.DataFrame(rows)
    
    # Example: Analytics with polars
    with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
        df = to_polars(db, "SELECT name, age, salary, active FROM test_basic.simple_table", limit=500)
        
        print(f"polars DataFrame shape: {df.shape}")
        print(f"\nSchema:\n{df.schema}")
        
        # Example: aggregate statistics
        if 'salary' in df.columns:
            summary = df.select([
                pl.col("salary").mean().alias("avg_salary"),
                pl.col("salary").max().alias("max_salary"),
                pl.col("salary").min().alias("min_salary"),
                pl.col("age").mean().alias("avg_age"),
            ])
            print(f"\nSummary statistics:\n{summary}")
else:
    print("Skipping polars example - polars not installed")
    print("Install with: pip install polars")

In [ ]:
# 2.3 Basic Data Exploration Patterns (no external dependencies)
# Use case: Quick data exploration without pandas/polars

def explore_table(db, keyspace: str, table: str, sample_size: int = 10):
    """Quick exploration of a table's structure and sample data."""
    query = f"SELECT * FROM {keyspace}.{table}"
    
    # Get sample
    result = db.execute(f"{query} LIMIT {sample_size}")
    
    if not result.rows:
        print(f"Table {keyspace}.{table} is empty or inaccessible")
        return
    
    # Column analysis
    first_row = result.rows[0]
    columns = first_row.keys()
    
    print(f"Table: {keyspace}.{table}")
    print(f"Sample size: {len(result.rows)} rows")
    print(f"Columns: {len(columns)}")
    print("-" * 60)
    
    # Analyze each column
    for col in list(columns)[:10]:  # Limit to first 10 columns
        values = [row.get(col) for row in result.rows]
        non_null = [v for v in values if v is not None]
        
        type_name = type(non_null[0]).__name__ if non_null else "unknown"
        null_count = len(values) - len(non_null)
        
        print(f"  {col:25} {type_name:15} nulls={null_count}")
        
        # Show sample value
        if non_null:
            sample = non_null[0]
            sample_str = repr(sample)
            if len(sample_str) > 40:
                sample_str = sample_str[:40] + "..."
            print(f"    sample: {sample_str}")
    
    if len(columns) > 10:
        print(f"  ... and {len(columns) - 10} more columns")

# Example: Explore multiple tables
print("=" * 60)
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    explore_table(db, "test_basic", "simple_table", sample_size=5)

print("\n" + "=" * 60)
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_COLLECTIONS)) as db:
    explore_table(db, "test_collections", "collection_table", sample_size=3)

## 3. Batch Processing

Patterns for processing large datasets efficiently with progress reporting and memory management.

In [ ]:
# 3.1 Process All Partitions
# Use case: ETL pipelines, data transformation, aggregation

def process_partitions(db, query: str, processor: Callable, batch_size: int = 100) -> list:
    """Process query results in batches with a custom processor function.
    
    Args:
        db: Database connection
        query: CQL query
        processor: Function(batch: list[Row]) -> Any
        batch_size: Number of rows per batch
    
    Returns:
        List of processor results for each batch
    """
    results = []
    batch = []
    
    for row in db.execute_streaming(query):
        batch.append(row)
        
        if len(batch) >= batch_size:
            result = processor(batch)
            results.append(result)
            batch = []
    
    # Process remaining rows
    if batch:
        result = processor(batch)
        results.append(result)
    
    return results

# Example processor: calculate average salary per batch
def calc_avg_salary(batch) -> dict:
    """Example processor: calculate average salary."""
    salaries = [row.get('salary') for row in batch if row.get('salary') is not None]
    return {
        'batch_size': len(batch),
        'avg_salary': sum(salaries) / len(salaries) if salaries else 0,
        'max_salary': max(salaries) if salaries else 0
    }

with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    batch_results = process_partitions(
        db,
        "SELECT * FROM test_basic.simple_table",
        calc_avg_salary,
        batch_size=200
    )
    
    print(f"Processed {len(batch_results)} batches")
    for i, result in enumerate(batch_results[:3]):
        print(f"  Batch {i}: size={result['batch_size']}, avg=${result['avg_salary']:,.0f}")
    
    # Overall statistics
    total_rows = sum(r['batch_size'] for r in batch_results)
    overall_avg = sum(r['avg_salary'] * r['batch_size'] for r in batch_results) / total_rows
    print(f"\nTotal rows: {total_rows:,}")
    print(f"Overall average salary: ${overall_avg:,.2f}")

In [ ]:
# 3.2 Memory-Efficient Iteration
# Use case: Processing very large tables that don't fit in memory

def memory_efficient_count(db, query: str) -> dict:
    """Count rows and track memory usage during streaming.
    
    Demonstrates that memory stays bounded regardless of table size.
    """
    tracemalloc.start()
    
    count = 0
    type_counts = {}
    peak_memory = 0
    
    config = cqlite.StreamingConfig(buffer_size=256, chunk_size=1000)
    
    for row in db.execute_streaming(query, config=config):
        count += 1
        
        # Example processing: count by a column value
        active = row.get('active')
        if active is not None:
            key = 'active' if active else 'inactive'
            type_counts[key] = type_counts.get(key, 0) + 1
        
        # Sample memory periodically
        if count % 500 == 0:
            current, peak = tracemalloc.get_traced_memory()
            peak_memory = max(peak_memory, peak)
    
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    return {
        'total_rows': count,
        'type_counts': type_counts,
        'peak_memory_mb': peak / 1024 / 1024
    }

# Example: Memory-efficient processing
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    stats = memory_efficient_count(db, "SELECT * FROM test_basic.simple_table")
    
    print(f"Total rows processed: {stats['total_rows']:,}")
    print(f"Active/Inactive breakdown: {stats['type_counts']}")
    print(f"Peak memory usage: {stats['peak_memory_mb']:.2f} MB")
    print("\nMemory stayed bounded - streaming works!")

In [ ]:
# 3.3 Progress Reporting for Long Operations
# Use case: ETL jobs, data migrations, batch updates

class ProgressReporter:
    """Simple progress reporter for long-running operations."""
    
    def __init__(self, total_estimate: int = None, report_interval: int = 100):
        self.total_estimate = total_estimate
        self.report_interval = report_interval
        self.processed = 0
        self.start_time = time.time()
        self.errors = 0
    
    def update(self, count: int = 1, error: bool = False):
        self.processed += count
        if error:
            self.errors += 1
        
        if self.processed % self.report_interval == 0:
            elapsed = time.time() - self.start_time
            rate = self.processed / elapsed if elapsed > 0 else 0
            
            if self.total_estimate:
                pct = self.processed / self.total_estimate * 100
                eta = (self.total_estimate - self.processed) / rate if rate > 0 else 0
                print(f"  Progress: {self.processed:,}/{self.total_estimate:,} ({pct:.1f}%) "
                      f"- {rate:.0f} rows/sec - ETA: {eta:.0f}s", end='\r')
            else:
                print(f"  Processed: {self.processed:,} - {rate:.0f} rows/sec", end='\r')
    
    def summary(self):
        elapsed = time.time() - self.start_time
        rate = self.processed / elapsed if elapsed > 0 else 0
        print(f"\n{'='*60}")
        print(f"Completed: {self.processed:,} rows in {elapsed:.1f}s ({rate:.0f} rows/sec)")
        if self.errors:
            print(f"Errors: {self.errors}")

# Example: ETL job with progress reporting
def etl_job(db, source_query: str, transform_fn: Callable, estimated_rows: int = None) -> list:
    """Example ETL job with progress reporting."""
    progress = ProgressReporter(estimated_rows, report_interval=200)
    
    results = []
    
    for row in db.execute_streaming(source_query):
        try:
            # Transform
            transformed = transform_fn(row)
            results.append(transformed)
            progress.update()
        except Exception as e:
            progress.update(error=True)
    
    progress.summary()
    return results

# Example transformation
def normalize_user(row) -> dict:
    """Transform user record: normalize name, calculate age bracket."""
    name = row.get('name', '') or ''
    name = name.strip() if isinstance(name, str) else ''
    age = row.get('age', 0) or 0
    
    return {
        'name': name.title() if name else 'Unknown',
        'age_bracket': 'young' if age < 30 else 'middle' if age < 50 else 'senior',
        'salary_k': (row.get('salary', 0) or 0) // 1000
    }

with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    transformed = etl_job(
        db,
        "SELECT name, age, salary FROM test_basic.simple_table",
        normalize_user,
        estimated_rows=1000  # Approximate
    )
    
    # Show sample
    print(f"\nSample transformed records:")
    for record in transformed[:3]:
        print(f"  {record}")

## 4. Error Handling Patterns

Robust patterns for handling errors gracefully in production environments.

In [ ]:
# 4.1 Graceful Recovery from Errors
# Use case: Production ETL pipelines that must continue despite errors

def safe_query(db, query: str, default=None):
    """Execute a query with graceful error handling.
    
    Returns:
        QueryResult on success, default value on error
    """
    try:
        return db.execute(query)
    except cqlite.ParseError as e:
        print(f"Query syntax error: {e}")
        return default
    except cqlite.QueryError as e:
        print(f"Query execution error: {e}")
        return default
    except cqlite.CqliteError as e:
        print(f"CQLite error: {e}")
        return default

def query_with_retry(db, query: str, max_retries: int = 3, 
                     backoff_factor: float = 0.1):
    """Execute query with retry logic.
    
    Useful for handling transient failures (though rare with local SSTables).
    """
    last_error = None
    
    for attempt in range(max_retries):
        try:
            return db.execute(query)
        except cqlite.CqliteError as e:
            last_error = e
            if attempt < max_retries - 1:
                sleep_time = backoff_factor * (2 ** attempt)
                print(f"Attempt {attempt + 1} failed, retrying in {sleep_time}s...")
                time.sleep(sleep_time)
    
    raise last_error

# Example: Safe queries
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    # Valid query
    result = safe_query(db, "SELECT * FROM test_basic.simple_table LIMIT 5")
    if result:
        print(f"Valid query returned {len(result.rows)} rows")
    
    # Invalid query (syntax error)
    result = safe_query(db, "SELEKT * FROM test_basic.simple_table", default=[])
    print(f"Invalid query returned: {result}")
    
    # Non-existent table (may return empty or error depending on implementation)
    result = safe_query(db, "SELECT * FROM nonexistent.table", default=[])
    print(f"Missing table returned: {result}")

In [ ]:
# 4.2 Validation Patterns
# Use case: Data quality checks, migration validation

def validate_table(db, keyspace: str, table: str, 
                   validators: list) -> dict:
    """Run validation checks on a table.
    
    Args:
        validators: List of (name, validator_fn) tuples
                   validator_fn(rows) -> (passed: bool, message: str)
    """
    query = f"SELECT * FROM {keyspace}.{table}"
    
    try:
        result = db.execute(query)
    except cqlite.CqliteError as e:
        return {'table': f"{keyspace}.{table}", 'error': str(e), 'validations': []}
    
    rows = list(result.rows)
    validation_results = []
    
    for name, validator in validators:
        try:
            passed, message = validator(rows)
            validation_results.append({
                'check': name,
                'passed': passed,
                'message': message
            })
        except Exception as e:
            validation_results.append({
                'check': name,
                'passed': False,
                'message': f"Validator error: {e}"
            })
    
    return {
        'table': f"{keyspace}.{table}",
        'row_count': len(rows),
        'validations': validation_results,
        'all_passed': all(v['passed'] for v in validation_results)
    }

# Example validators
def check_no_empty_names(rows):
    """Validate that no names are empty."""
    empty = [r for r in rows if not r.get('name')]
    if empty:
        return False, f"Found {len(empty)} rows with empty names"
    return True, "All names are populated"

def check_positive_salaries(rows):
    """Validate that salaries are positive."""
    invalid = [r for r in rows if (r.get('salary') or 0) < 0]
    if invalid:
        return False, f"Found {len(invalid)} negative salaries"
    return True, "All salaries are non-negative"

def check_valid_ages(rows):
    """Validate that ages are in reasonable range."""
    invalid = [r for r in rows if not (0 <= (r.get('age') or 0) <= 150)]
    if invalid:
        return False, f"Found {len(invalid)} invalid ages"
    return True, "All ages are valid"

# Run validations
with cqlite.open(str(DATA_DIR), schema=str(SCHEMA_BASIC)) as db:
    validators = [
        ('non_empty_names', check_no_empty_names),
        ('positive_salaries', check_positive_salaries),
        ('valid_ages', check_valid_ages),
    ]
    
    report = validate_table(db, 'test_basic', 'simple_table', validators)
    
    print(f"Validation Report: {report['table']}")
    print(f"Row count: {report['row_count']}")
    print(f"Overall: {'PASSED' if report['all_passed'] else 'FAILED'}")
    print("-" * 50)
    for v in report['validations']:
        status = 'PASS' if v['passed'] else 'FAIL'
        print(f"  [{status}] {v['check']}: {v['message']}")

## 5. Multi-table Operations

Patterns for working with multiple tables and keyspaces.

In [ ]:
# 5.1 Query Across Multiple Keyspaces
# Use case: Aggregating data from different schemas

def multi_keyspace_summary(data_dir: Path, keyspace_configs: list) -> dict:
    """Query multiple keyspaces and aggregate results.
    
    Args:
        keyspace_configs: List of (keyspace, table, schema_path, query) tuples
    
    Returns:
        Dictionary with results from each keyspace
    """
    results = {}
    
    for keyspace, table, schema_path, query in keyspace_configs:
        key = f"{keyspace}.{table}"
        try:
            with cqlite.open(str(data_dir), schema=str(schema_path)) as db:
                result = db.execute(query)
                results[key] = {
                    'row_count': len(result.rows),
                    'sample': result.rows[0].to_dict() if result.rows else None,
                    'status': 'success'
                }
        except cqlite.CqliteError as e:
            results[key] = {
                'row_count': 0,
                'sample': None,
                'status': f'error: {e}'
            }
    
    return results

# Example: Query all keyspaces
configs = [
    ('test_basic', 'simple_table', SCHEMA_BASIC, 
     "SELECT id, name, age FROM test_basic.simple_table LIMIT 5"),
    ('test_collections', 'collection_table', SCHEMA_COLLECTIONS,
     "SELECT id, tags FROM test_collections.collection_table LIMIT 5"),
    ('test_timeseries', 'sensor_data', SCHEMA_TIMESERIES,
     "SELECT sensor_id, temperature FROM test_timeseries.sensor_data LIMIT 5"),
    ('test_wide_rows', 'product_catalog', SCHEMA_WIDE_ROWS,
     "SELECT product_id, product_name FROM test_wide_rows.product_catalog LIMIT 5"),
]

summary = multi_keyspace_summary(DATA_DIR, configs)

print("Multi-Keyspace Summary")
print("=" * 60)
for table_name, info in summary.items():
    print(f"\n{table_name}:")
    print(f"  Status: {info['status']}")
    print(f"  Rows: {info['row_count']}")
    if info['sample']:
        sample_keys = list(info['sample'].keys())[:3]
        print(f"  Sample columns: {sample_keys}")

In [ ]:
# 5.2 Comparing Data Between Tables
# Use case: Data migration verification, consistency checks

def compare_row_counts(data_dir: Path, comparisons: list) -> list:
    """Compare row counts between related tables.
    
    Args:
        comparisons: List of (table1_config, table2_config, description) tuples
                    where config = (keyspace, table, schema_path)
    """
    results = []
    
    for config1, config2, description in comparisons:
        ks1, tbl1, schema1 = config1
        ks2, tbl2, schema2 = config2
        
        # Get count from table 1
        try:
            with cqlite.open(str(data_dir), schema=str(schema1)) as db:
                r1 = db.execute(f"SELECT * FROM {ks1}.{tbl1}")
                count1 = len(r1.rows)
        except cqlite.CqliteError:
            count1 = None
        
        # Get count from table 2
        try:
            with cqlite.open(str(data_dir), schema=str(schema2)) as db:
                r2 = db.execute(f"SELECT * FROM {ks2}.{tbl2}")
                count2 = len(r2.rows)
        except cqlite.CqliteError:
            count2 = None
        
        results.append({
            'description': description,
            'table1': f"{ks1}.{tbl1}",
            'count1': count1,
            'table2': f"{ks2}.{tbl2}",
            'count2': count2,
            'match': count1 == count2 if count1 is not None and count2 is not None else None
        })
    
    return results

# Example: Compare related tables
comparisons = [
    (
        ('test_basic', 'simple_table', SCHEMA_BASIC),
        ('test_basic', 'uncompressed_table', SCHEMA_BASIC),
        "simple_table vs uncompressed_table"
    ),
    (
        ('test_collections', 'collection_table', SCHEMA_COLLECTIONS),
        ('test_collections', 'frozen_collections_table', SCHEMA_COLLECTIONS),
        "collection_table vs frozen_collections_table"
    ),
]

comparison_results = compare_row_counts(DATA_DIR, comparisons)

print("Table Comparison Results")
print("=" * 60)
for r in comparison_results:
    match_str = "MATCH" if r['match'] else "DIFFER" if r['match'] is False else "ERROR"
    print(f"\n{r['description']}:")
    print(f"  {r['table1']}: {r['count1']} rows")
    print(f"  {r['table2']}: {r['count2']} rows")
    print(f"  Status: {match_str}")

## Summary

This notebook demonstrated key patterns for real-world CQLite usage:

### Data Export
- **CSV**: Universal format for spreadsheets and data tools
- **JSONL**: Streaming-friendly format for pipelines
- **Parquet**: Columnar format for data lakes (requires pyarrow)
- **Streaming export**: Memory-efficient for large datasets

### Analytics Integration
- **pandas**: Full-featured data analysis
- **polars**: High-performance alternative
- **Pure Python**: Works without external dependencies

### Batch Processing
- **Partition processing**: Handle data in manageable batches
- **Memory efficiency**: Streaming keeps memory bounded
- **Progress reporting**: Track long-running operations

### Error Handling
- **Graceful recovery**: Continue despite individual failures
- **Validation patterns**: Data quality checks
- **Retry logic**: Handle transient issues

### Multi-table Operations
- **Cross-keyspace queries**: Aggregate from multiple schemas
- **Data comparison**: Verify consistency between tables

### Next Steps
- Explore the full API in `acceptance-testing.ipynb`
- Check the CQLite documentation for advanced features
- See the test suite for more examples